<img src="logo.png" alt="Vegeta" width="240">

# Quadcopter — frame, propeller, noise, vibration and life

One notebook for one machine. Part 1 designs the frame (CAD → FEA → CFD → print, with a revision
history in a workspace). Part 2 sizes the propeller and the drive (blade element theory, a rotating-frame
CFD check, excitation lines, noise). Part 3 takes the preferred frame through modal analysis, a Campbell
diagram, three mission types, rainflow spectra, fatigue on the FEA stress fields and a fleet-life
simulation. Values pass from part to part as live variables; every assumption stays visible where it is
used. The OpenFOAM cells run when you run them (`VEGETA_SKIP_OPENFOAM=1` skips them in headless execution).

```
Part 1  frame:      mass budget → parametric CAD → load cases (FEA) → three revisions → forward-flight CFD → print
Part 2  propeller:  BEMT performance → motor + battery → excitations → noise → CFD check → JSON hand-off
Part 3  life:       modes + Campbell → unit stress fields → missions → spectra → damage → fleet life
```

# Part 1 — the frame

In [ ]:
import json, math, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos, aeromant, mellonia, core, boreas, chronos
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.mellonia import viz as mviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM
from vegeta.dedalus.examples import Propeller as PropellerCAD

ROOT = Path("_runs/quadcopter"); shutil.rmtree(ROOT, ignore_errors=True)
RUNS = ROOT / "frame"; RUNS.mkdir(parents=True)
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"

## 1. Mission and mass budget

A freestyle/utility quad: 4S battery, 5-inch props, all-up weight (AUW) below ~500 g, thrust-to-weight
≥ 4 at full throttle. Component masses are datasheet-style values you would take from the parts you
actually order; the frame mass is filled in later from the CAD volume.

In [ ]:
parts = pd.DataFrame([
    ("motors 2306 (4x)",        4 * 30.0),
    ("propellers 5x4.3 (4x)",   4 * 5.0),
    ("4-in-1 ESC",              15.0),
    ("flight controller",       10.0),
    ("battery 4S 1500 mAh",     180.0),
    ("camera + VTX + antenna",  35.0),
    ("receiver, wiring, bolts", 30.0),
], columns=["part", "mass_g"]).set_index("part")

MAX_THRUST_PER_MOTOR_N = 8.0      # ~815 gf per motor at full throttle on 4S with a 5x4.3 prop (datasheet-style value)
G = 9.81
parts

## 2. Parametric CAD (Dedalus)

The frame is one fused solid: a round centre plate with the flight-controller stack holes (30.5 mm),
four tapered arms at 45°, and round motor pads with a 16×16 mm M3 pattern. Loads and supports will
attach to the **bolt-hole cylinders** — the faces a real motor or stack actually pulls on — so no
special "load face" is needed in the CAD. An optional elliptic canopy encloses the electronics for the CFD.

A design note from the first attempt: with 26 mm pads the M3 holes left a 0.1 mm wall at the pad edge —
CadQuery builds it happily, Gmsh cannot mesh it. The design now checks the wall thickness itself.

The design lives in its own file (`designs/quad_frame.py`, copied into `_runs/`) so it can be revised, diffed and (notebook 07) handed to the AI copilot.

In [ ]:
design_file = RUNS / "quad_frame.py"
shutil.copy(Path("designs/quad_frame.py"), design_file)     # the original stays untouched; this copy may be edited
frame_design = dedalus.load_design(f"{design_file}:QuadFrame")
pd.DataFrame(frame_design.params.table()).set_index("name")

In [ ]:
frame = frame_design.generate()
frame          # interactive CadQuery view

In [ ]:
FRAME_DENSITY_G_MM3 = 1.25e-3     # PETG-CF, nominal (1.25 g/cm^3)
frame_mass_g = frame.volume * FRAME_DENSITY_G_MM3
parts.loc["frame (printed, from CAD volume)"] = round(frame_mass_g, 1)
AUW_g = parts["mass_g"].sum()
hover_thrust_per_motor_N = AUW_g / 1000 * G / 4
print(f"frame {frame_mass_g:.1f} g  |  AUW {AUW_g:.0f} g  |  hover thrust/motor {hover_thrust_per_motor_N:.2f} N  "
      f"|  thrust-to-weight {4 * MAX_THRUST_PER_MOTOR_N / (AUW_g / 1000 * G):.2f}")
parts

In [ ]:
dviz.show(dviz.plot3d(frame, show_edges=False))

In [ ]:
p0 = frame_design.resolve()
R = p0["wheelbase"] / 2
cx = R * math.cos(math.radians(45))
fig = dviz.plot_sections(frame, normal="x", positions=[0.0, 40.0, cx], cols=3)      # plate, arm, motor pad
fig = dviz.plot_sections(frame, normal="z", positions=[1.0, 3.0, 5.0], cols=3)      # in-plane cuts

## 3. Structural load cases (Talos)

Two explicit load cases, both with the frame bolted to the FC stack (the four stack-hole cylinders are
fixed); thrust enters through the four motor bolt holes of each pad:

| case | load | what it represents |
|---|---|---|
| `max_thrust` | +8 N up on each motor's bolt holes | full-throttle punch-out, symmetric |
| `hard_landing` | +40 N up on **one** motor's bolt holes | landing on one arm — a *chosen* design load, 5× the motor's max thrust |

Material: PETG-CF, nominal values (E = 4.8 GPa, ν = 0.38, yield 45 MPa). Printed parts are anisotropic;
these numbers are a starting point for comparison between revisions, not a certification.

Regions are selected geometrically from the parameters (boxes around each bolt pattern),
so the same factory works for every revision. `talos.inspect_step` shows the surface list if you want
to check what was selected.

In [ ]:
PETG_CF = talos.Material("PETG-CF", youngs_modulus=4800.0, poissons_ratio=0.38, density=1.25e-9,
                         yield_strength=45.0, source="nominal filament datasheet values, XY orientation")

def frame_regions(p):
    """Bolt-hole cylinders of each motor (motor0..3) and of the stack (stack0..3), by bounding box."""
    R, m, r = p["wheelbase"] / 2, p["motor_pattern"] / 2, p["motor_hole"] / 2 + 0.2
    regions = []
    for k in range(4):
        a = math.radians(45 + 90 * k)
        cx, cy = R * math.cos(a), R * math.sin(a)
        regions.append(talos.SurfacesInBox(f"motor{k}", (cx - m - r, cy - m - r, -0.1, cx + m + r, cy + m + r, 100.0)))
    s, r = p["stack_pattern"] / 2, p["stack_hole"] / 2 + 0.2
    for k, (sx, sy) in enumerate([(s, s), (-s, s), (s, -s), (-s, -s)]):
        regions.append(talos.SurfacesInBox(f"stack{k}", (sx - r, sy - r, -0.1, sx + r, sy + r, 100.0)))
    return regions

def frame_model(step, p, loads, name):
    return talos.StructuralModel(step, "mm-N-MPa", PETG_CF, frame_regions(p),
                                 supports=[talos.FixedSupport(f"stack{k}") for k in range(4)], loads=loads,
                                 mesh_settings=talos.MeshSettings(element_size=4.0), name=name)

def max_thrust(rev):
    return frame_model(rev.step, rev.params, [talos.Force(f"motor{k}", fz=MAX_THRUST_PER_MOTOR_N) for k in range(4)], "max_thrust")

def hard_landing(rev):
    return frame_model(rev.step, rev.params, [talos.Force("motor0", fz=40.0)], "hard_landing")

In [ ]:
frame.export_step(RUNS / "preview.step")
talos.inspect_step(RUNS / "preview.step", units="mm-N-MPa")

## 4. A workspace with revisions (vegeta.core)

Every revision is immutable: parameters, source hash, geometry and evaluations are recorded once.
Revision r1 is the baseline; we will branch from it.

In [ ]:
ws = core.Workspace.create(RUNS / "workspace", name="quad frame study")
quad = ws.add_design("frame", f"{design_file}:QuadFrame")
r1 = quad.new_revision(note="baseline: 12x6 arms, taper 0.7")
r1.generate()
ws.status()

In [ ]:
ev_t = r1.run_fea("max_thrust", max_thrust, progress=True)
ev_l = r1.run_fea("hard_landing", hard_landing, progress=True)
ws.status()

In [ ]:
mesh_res, solve_res = ev_t.tool_results[0], ev_t.tool_results[-1]
tviz.show(tviz.plot_problem(max_thrust(r1), mesh_res))

In [ ]:
if ev_t.ok:
    tviz.show(tviz.plot_results(solve_res, field="von_mises"))

In [ ]:
if ev_t.ok:
    fig = tviz.plot_section(solve_res, normal="z", origin=(0, 0, 3.0), field="von_mises")   # mid-thickness map
    fig = talos.plot_deformed(solve_res)

In [ ]:
if ev_l.ok:
    tviz.show(tviz.plot_results(ev_l.tool_results[-1], field="|U|"))
    print(ev_l)

### Reading the numbers
`max_displacement` is the tip deflection of a pad; `safety_factor_yield` is yield / peak nodal von Mises.
The peak sits at a bolt hole (support/load singularity) — compare it *between revisions*, do not read it
as an absolute. Reactions are returned per stack hole; CalculiX's reaction totals exclude loads on
supported nodes (none here).

## 5. Iterate: three candidate frames

The baseline arms are floppy under a hard landing. Two classic fixes, as branches of r1:
taller arms (stiffness ∝ h³) and wider, less tapered arms (more mass, more damage tolerance).
Every branch is generated and analysed with **exactly the same factories** — that is the point of
recording the load cases as code.

In [ ]:
def branch_once(parent, note, **params):
    # re-running this cell reuses the revision instead of creating a new one each time
    return next((r for r in ws.revisions() if r.record.get("note") == note), None) or parent.branch(note=note, **params)

r2 = branch_once(r1, "taller arms", arm_height=8.0, plate_thickness=8.0)
r3 = branch_once(r1, "wider arms, stronger taper", arm_width=16.0, taper=0.55)

# 4 FEA jobs (~25-35k second-order tets each): about a minute on a laptop, longer on a slow one.
# Safe to interrupt and re-run: finished work is skipped, an interrupted evaluation is moved aside.
jobs = [(r, case) for r in (r2, r3) for case in (max_thrust, hard_landing)]
for r, case in tqdm(jobs, desc="revisions x load cases"):
    if not r.is_generated:
        r.generate()
    if r.evaluation("fea", case.__name__) is None:
        r.run_fea(case.__name__, case, progress=True)
ws.status()

In [ ]:
def row(rev):
    vol = rev.geometry_summary()["metrics"]["volume"]
    d = {"rev": rev.id, "note": rev.record.get("note", ""), "arm_h": rev.params["arm_height"],
         "arm_w": rev.params["arm_width"], "taper": rev.params["taper"], "mass_g": vol * FRAME_DENSITY_G_MM3}
    for case in ("max_thrust", "hard_landing"):
        ev = rev.evaluation("fea", case)
        d[f"{case}_defl_mm"] = ev.metrics.get("max_displacement", np.nan) if ev and ev.ok else np.nan
        d[f"{case}_SF"] = ev.metrics.get("safety_factor_yield", np.nan) if ev and ev.ok else np.nan
    return d

table = pd.DataFrame([row(r) for r in (r1, r2, r3)]).set_index("rev").round(2)
table

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
table.plot.bar(y=["max_thrust_defl_mm", "hard_landing_defl_mm"], ax=ax[0], title="pad deflection [mm]")
ax[1].scatter(table["mass_g"], table["hard_landing_SF"], s=80)
for rid, r in table.iterrows():
    ax[1].annotate(rid, (r["mass_g"], r["hard_landing_SF"]), textcoords="offset points", xytext=(6, 4))
ax[1].set(xlabel="frame mass [g]", ylabel="SF, hard landing", title="mass vs safety factor"); ax[1].grid(alpha=0.3)
fig.tight_layout()

In [ ]:
best = table["hard_landing_SF"].idxmax()
preferred = ws.revision(best)
preferred.label("preferred", note=f"best hard-landing SF ({table.loc[best, 'hard_landing_SF']}) at {table.loc[best, 'mass_g']} g")
for r in (r1, r2, r3):
    if r.id != best:
        r.label("rejected", note="lower safety factor than the preferred revision")
ws.status()

## 6. Forward flight drag with a canopy (Aeromant)

The preferred frame gets an elliptic canopy over the electronics. We run steady RANS (k-ω SST) at
15 m/s with the frame at 0° pitch (a real quad flies nose-down; that is a parameter for the next
revision), coarse mesh for a *comparative* drag figure. Reference area = plate footprint.

The CFD evaluation is skipped when `VEGETA_SKIP_OPENFOAM=1` is set; the later parts do not depend on it.

In [ ]:
r4 = preferred.branch(canopy_height=25.0, note="canopy for CFD")
r4.generate(stl_tolerance=0.1)
cases = {}

def forward_flight(rev, workdir):
    p = rev.params
    case = aeromant.CFDCase(
        "rans_ksst_external", rev.stl,
        dict(velocity=15.0, kinematic_viscosity=1.5e-5, density=1.2,
             reference_area=(p["plate_size"] / 1000) ** 2, reference_length=0.1, center_of_rotation=(0, 0, 0),
             iterations=250, surface_level=3, near_level=2, wake_level=1, cells_per_length=2.0),
        workdir=workdir, geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())
    cases[rev.id] = case
    return case

ev_cfd = r4.run_cfd("forward_15ms", forward_flight, progress=True) if RUN_CFD else None
print(ev_cfd if ev_cfd is not None else "CFD skipped (VEGETA_SKIP_OPENFOAM=1): run this cell on a machine with OpenFOAM")

In [ ]:
if ev_cfd is not None:
    case = cases[r4.id]
    aviz.show(aviz.plot_setup(case))

In [ ]:
if ev_cfd is not None and ev_cfd.ok:
    aviz.show(aviz.plot_mesh_slice(case, normal="y"))

In [ ]:
if ev_cfd is not None and ev_cfd.ok:
    aviz.show(aviz.plot_field_slice(case, "U", normal="y"))
    fig = aviz.plot_section(case, "p", normal="y", zoom=2)

In [ ]:
if ev_cfd is not None and ev_cfd.ok:
    aviz.show(aviz.plot_streamlines(case))

In [ ]:
if ev_cfd is not None and ev_cfd.ok:
    aviz.show(aviz.plot_surface_pressure(case))
    F = ev_cfd.metrics["drag_force_N"]
    print(f"Cd {ev_cfd.metrics['Cd']:.3f}  drag {F:.2f} N at 15 m/s  ->  parasite power {F * 15:.1f} W "
          f"(rotor induced power dominates; this is the airframe share)")

## 7. Print the preferred frame (Mellonia)

Flat on the bed, no supports needed. The example profile is generic PLA — swap in
your printer's exported PETG-CF profile with `PrintSettings.from_ini(...)`.

In [ ]:
ev_p = preferred.run_print("flat", GENERIC_PLA_0_2MM, mellonia.Orientation())
print(ev_p)

In [ ]:
if ev_p.ok:
    prn = ev_p.tool_results[0]
    mviz.show(mviz.plot_toolpath(prn))

In [ ]:
if ev_p.ok:
    fig = mviz.plot_layer_grid(prn, n=6, cols=3)

## 8. Where we are

The workspace is the record: each revision folder holds `revision.json`, the STEP/STL, and one
directory per evaluation with the tools' native files (`mesh.msh`, `.inp`, `.frd`, the OpenFOAM case,
the G-code) plus every command that was run.

In [ ]:
parts.loc["frame (printed, from CAD volume)"] = round(table.loc[best, "mass_g"], 1)
print(f"AUW {parts['mass_g'].sum():.0f} g with the preferred frame ({best})")
ws.status()

In [ ]:
for p in sorted((RUNS / "workspace" / "revisions" / best).rglob("*"))[:30]:
    print(p.relative_to(RUNS / "workspace"))

**Next steps an engineer would take:** pitch the frame in the CFD (`rotate` the STL or add a `pitch`
parameter), add the landing-gear and arm-break-away features, run the hard-landing case with the
printed-part anisotropy (a second material with lower Z properties), and let the AI copilot
(notebook 07) propose weight-saving cut-outs on `_runs/quad/quad_frame.py` — every proposal is built and
measured before you accept it.

# Part 2 — the propeller and the drive

In [ ]:
RUNS = ROOT / "propeller"; RUNS.mkdir()
RHO = 1.2
AUW_KG = parts["mass_g"].sum() / 1000          # all-up weight with the preferred frame (Part 1 mass budget)
MOTORS = 4
HOVER_THRUST_PER_MOTOR = AUW_KG * 9.81 / MOTORS
print(f"AUW {AUW_KG:.3f} kg -> hover thrust {HOVER_THRUST_PER_MOTOR:.2f} N per motor")

## 1. The hardware (your inputs)

In [ ]:
d, p = boreas.inches(5, 4.3)
prop = boreas.Propeller.from_pitch("5x4.3 tri-blade", d, p, blades=3, chord_root_m=0.010, chord_max_m=0.016,
                                   chord_tip_m=0.006, mass_kg=0.0045, rotor_mass_kg=0.020,   # rotor = prop + motor bell
                                   notes="generic planform; fit chord/beta to the real propeller for better numbers")
airfoil = boreas.Airfoil(name="thin cambered section", cl_alpha=2 * math.pi * 0.9, alpha0_deg=-3.0, cl_max=1.1, cd0=0.025, k=0.045,
                         source="assumed for a moulded 5-inch blade at Re ~ 1e5")
motor = boreas.Motor("2306-2400KV", kv_rpm_per_volt=2400, resistance_ohm=0.06, no_load_current_a=1.2, max_current_a=40, mass_kg=0.030)
battery = boreas.Battery("4S 1500 mAh", cells=4, capacity_ah=1.5, usable_fraction=0.8, mass_kg=0.180)
system = boreas.Propulsion(prop, airfoil, motor, battery, rho=RHO)
pd.DataFrame(prop.describe()["stations"]).set_index("r_m").T

## 2. The propeller, drawn

Planform and blade angle from the station table, and the same planform built as a solid in Dedalus
(`vegeta.dedalus.examples.Propeller` uses the same formula): sections at three radii, a 3D view, and an
STL export for a later CFD or print.

In [ ]:
r = np.array(prop.r); c = np.array(prop.chord); beta = np.array(prop.beta_deg)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].fill_between(r * 1000, -c * 1000 * 0.3, c * 1000 * 0.7, color="#9fb8d0"); ax[0].set_aspect("equal")
ax[0].set(xlabel="radius [mm]", ylabel="chord [mm]", title=f"planform, solidity {prop.solidity:.3f}"); ax[0].grid(alpha=0.3)
ax[1].plot(r * 1000, beta, "o-"); ax[1].set(xlabel="radius [mm]", ylabel="blade angle β [deg]", title=f"twist for {p * 1000:.0f} mm pitch"); ax[1].grid(alpha=0.3)
fig.tight_layout()

In [ ]:
CAD_KW = dict(diameter=d * 1000, pitch=p * 1000, blades=3, hub_diameter=12, hub_height=7, bore=5, chord_root=10, chord_max=16, chord_tip=6, thickness=0.10, camber=0.05)
cad = PropellerCAD().generate(**CAD_KW)
prop_files = cad.export(RUNS / "quad_5x43_cad", formats=("step", "stl"), stl_tolerance=0.02)
print(f"CAD volume {cad.volume:.0f} mm^3 -> {cad.volume * 1.2e-3:.1f} g at 1.2 g/cm^3 (input mass_kg = {prop.mass_kg * 1000:.1f} g)")
dviz.show(dviz.plot3d(cad))

In [ ]:
fig = dviz.plot_sections(cad, normal="x", positions=[20.0, 40.0, 58.0], cols=3)   # blade sections along +X

## 3. Static performance (hover): thrust and power against rpm

In [ ]:
rpms = np.linspace(4000, 30000, 27)
static = [boreas.solve(prop, airfoil, rpm, 0.0, RHO) for rpm in rpms]
T = np.array([o.thrust for o in static]); P = np.array([o.power for o in static])
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(rpms, T); ax[0].axhline(HOVER_THRUST_PER_MOTOR, ls="--", color="#c62828", label="hover thrust / motor"); ax[0].legend()
ax[0].set(xlabel="rpm", ylabel="thrust [N]", title="static thrust")
ax[1].plot(rpms, P); ax[1].set(xlabel="rpm", ylabel="shaft power [W]", title="static power")
ax[2].plot(rpms, [o.figure_of_merit for o in static]); ax[2].set(xlabel="rpm", ylabel="figure of merit", ylim=(0, 1), title="hover efficiency")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()
print(f"at 20 000 rpm: Ct {static[np.argmin(abs(rpms - 20000))].ct:.4f}, Cp {static[np.argmin(abs(rpms - 20000))].cp:.4f}, "
      f"tip Mach {static[-1].tip_mach:.2f} at {rpms[-1]:.0f} rpm")

In [ ]:
hover_aero = boreas.rpm_for_thrust(prop, airfoil, HOVER_THRUST_PER_MOTOR, 0.0, RHO)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].plot(hover_aero.r * 1000, hover_aero.dT_dr); ax[0].set(xlabel="radius [mm]", ylabel="dT/dr [N/m]", title=f"thrust loading at hover ({hover_aero.rpm:.0f} rpm)")
ax[1].plot(hover_aero.r * 1000, hover_aero.alpha_deg); ax[1].set(xlabel="radius [mm]", ylabel="angle of attack [deg]", title="section incidence")
ax[2].plot(hover_aero.r * 1000, hover_aero.induced_velocity); ax[2].set(xlabel="radius [mm]", ylabel="induced velocity [m/s]", title="inflow")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()

## 4. Motor + battery: throttle → rpm, current, power

The motor curve meets the propeller torque curve at each throttle; current above the motor limit is
flagged, never hidden.

In [ ]:
throttles = np.linspace(0.2, 1.0, 17)
sweep = system.sweep(throttles, airspeed=0.0)
sw = pd.DataFrame([{"throttle": s.throttle, "rpm": s.rpm, "thrust_N": s.thrust, "current_A": s.current,
                    "electrical_W": s.electrical_power, "motor_eff": s.motor_efficiency, "current_limited": s.current_limited} for s in sweep])
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(sw.throttle, sw.thrust_N, "o-"); ax[0].axhline(HOVER_THRUST_PER_MOTOR, ls="--", color="#c62828"); ax[0].set(xlabel="throttle", ylabel="thrust [N]")
ax[1].plot(sw.throttle, sw.current_A, "o-"); ax[1].axhline(motor.max_current_a, ls="--", color="#c62828", label="motor limit"); ax[1].legend(); ax[1].set(xlabel="throttle", ylabel="current [A]")
ax[2].plot(sw.thrust_N, sw.electrical_W / sw.thrust_N, "o-"); ax[2].set(xlabel="thrust [N]", ylabel="W per N", title="electrical power per newton")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()
sw.round(3)

In [ ]:
hover = system.for_thrust(HOVER_THRUST_PER_MOTOR)
punch = system.at_throttle(1.0)
cruise = system.for_thrust(HOVER_THRUST_PER_MOTOR * 1.3)        # forward flight at ~40 deg tilt: 1/cos(40) ~ 1.3
hover_minutes = battery.usable_wh / (MOTORS * hover.electrical_power) * 60
summary = pd.DataFrame({
    "hover": {"throttle": hover.throttle, "rpm": hover.rpm, "thrust_N": hover.thrust, "current_A": hover.current, "electrical_W": hover.electrical_power},
    "cruise (tilted)": {"throttle": cruise.throttle, "rpm": cruise.rpm, "thrust_N": cruise.thrust, "current_A": cruise.current, "electrical_W": cruise.electrical_power},
    "full throttle": {"throttle": 1.0, "rpm": punch.rpm, "thrust_N": punch.thrust, "current_A": punch.current, "electrical_W": punch.electrical_power},
}).round(2)
print(f"thrust-to-weight at full throttle: {MOTORS * punch.thrust / (AUW_KG * 9.81):.2f}"
      + ("  (CURRENT LIMITED: the motor cannot deliver this; the ESC/motor limit decides)" if punch.current_limited else ""))
print(f"hover endurance: {hover_minutes:.1f} min on {battery.usable_wh:.0f} Wh usable")
summary

## 5. What the frame will feel: excitation frequencies and forces

Per motor: shaft frequency (1P), blade-pass frequency (3P for three blades) and the rotating unbalance
force for an ISO balance grade G 6.3 (a typical hobby-grade prop; G 2.5 after balancing). These are the
inputs for the vibration and fatigue notebooks: a frame mode near one of these lines is a problem.

In [ ]:
rr = np.linspace(5000, 30000, 26)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(rr, rr / 60, label="1P (shaft)"); ax[0].plot(rr, prop.blades * rr / 60, label=f"{prop.blades}P (blade pass)")
for name, pt in (("hover", hover), ("full", punch)):
    ax[0].axvline(pt.rpm, color="#888", ls=":"); ax[0].text(pt.rpm, 50, name, rotation=90, va="bottom")
ax[0].set(xlabel="rpm", ylabel="frequency [Hz]", title="excitation lines"); ax[0].legend(); ax[0].grid(alpha=0.3)
for g in (6.3, 2.5):
    ax[1].plot(rr, [boreas.unbalance_force(prop.rotor_mass_kg, x, g) for x in rr], label=f"G {g}")
ax[1].set(xlabel="rpm", ylabel="rotating force [N]", title=f"unbalance force, rotor {prop.rotor_mass_kg * 1000:.0f} g"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()
pd.DataFrame({k: boreas.excitations(prop, v.rpm) for k, v in (("hover", hover), ("cruise", cruise), ("full", punch))}).round(2)

### Noise: blade-passing tones and broadband

Gutin's steady-loading tones (thrust and torque on the blades) at 1 m broadside, dB re 20 µPa, plus an
empirical broadband allowance for the vortex/trailing-edge noise, added in power. Four rotors add
10 log10(4) ≈ 6 dB; distance takes 20 log10(d) off. Tones from unsteady loading (a body in the inflow,
rotor–rotor interaction) are not modelled; treat these as the floor.

In [ ]:
DIST, ANGLE = 1.0, 90.0
noise_rows = {}
for name, pt in (("hover", hover), ("cruise", cruise), ("full", punch)):
    tones = boreas.gutin_harmonics(prop, pt.thrust, pt.aero.torque, pt.rpm, DIST, ANGLE, boreas.AIR, harmonics=6)
    bb = boreas.broadband_level(prop, pt.thrust, pt.rpm, DIST, boreas.AIR)
    one = 10 * math.log10(10 ** (tones["total_tonal_db"] / 10) + 10 ** (bb / 10))
    noise_rows[name] = {"rpm": pt.rpm, "BPF_hz": tones["blade_pass_hz"], "tonal_dB": tones["total_tonal_db"], "broadband_dB": bb,
                        "one_rotor_dB_at_1m": one, f"{MOTORS}_rotors_dB_at_1m": one + 10 * math.log10(MOTORS),
                        f"{MOTORS}_rotors_dB_at_10m": one + 10 * math.log10(MOTORS) - 20, f"{MOTORS}_rotors_dB_at_50m": one + 10 * math.log10(MOTORS) - 20 * math.log10(50)}
    if name == "hover":
        fig, ax = plt.subplots(figsize=(6.5, 3.4))
        ax.bar(tones["frequency_hz"], tones["spl_db"], width=25, label="Gutin tones, hover, one rotor")
        ax.axhline(bb, color="#c62828", ls="--", label="broadband allowance"); ax.set(xlabel="frequency [Hz]", ylabel="dB re 20 µPa at 1 m"); ax.legend(); ax.grid(alpha=0.3)
noise = pd.DataFrame(noise_rows).T.round(1)
noise

## 6. Axial inflow: climbing and descending

Thrust falls with axial speed (climb); the map covers 0–15 m/s. A vertical descent runs into the
vortex-ring regime that momentum theory cannot describe, so the map stops at zero airspeed.

In [ ]:
grid = boreas.performance_map(prop, airfoil, np.linspace(5000, 30000, 11), np.linspace(0, 15, 6), RHO)
Tg = np.array(grid["thrust_n"])
fig, ax = plt.subplots(figsize=(6, 3.8))
for j, v in enumerate(grid["airspeed_m_s"]):
    ax.plot(grid["rpm"], Tg[:, j], label=f"{v:.0f} m/s")
ax.set(xlabel="rpm", ylabel="thrust [N]", title="thrust vs rpm at axial airspeeds"); ax.legend(ncol=2); ax.grid(alpha=0.3)
print("unconverged map points:", grid["unconverged_points"])

## 7. Export — the hand-off

One JSON with: propeller geometry, the section model, motor and battery, the rpm × airspeed map, and the
named operating points **hover / cruise / full** each with its excitation summary (rpm, 1P, blade-pass,
unbalance force). Later notebooks copy from this file by hand (mission segments, vibration loads); the
STL/STEP are for CFD or printing.

In [ ]:
res = boreas.export(RUNS / "quad_5x43.json", prop, airfoil, map=grid, motor=motor, battery=battery,
                    points={"hover": hover, "cruise": cruise, "full": punch},
                    notes=f"quadcopter from notebook 08; AUW {AUW_KG} kg, {MOTORS} motors; hover endurance {hover_minutes:.1f} min")
print(res)
doc = boreas.load(res.artifacts["json"])
pd.DataFrame({k: {"rpm": v["rpm"], "thrust_N": v["aero"]["thrust"], "current_A": v["current"],
                  "shaft_hz": v["excitation"]["shaft_hz"], "blade_pass_hz": v["excitation"]["blade_pass_hz"],
                  "unbalance_N": v["excitation"]["unbalance_force_n"]} for k, v in doc["points"].items()}).round(2)

In [ ]:
print("files for the next steps:")
for f in sorted(RUNS.glob("quad_5x43*")):
    print("  ", f, f"({f.stat().st_size / 1024:.0f} kB)" if f.is_file() else "")
print("\nkeys in the JSON:", list(doc))
print("points:", list(doc["points"]), "| map:", len(doc["map"]["rpm"]), "rpm x", len(doc["map"]["airspeed_m_s"]), "airspeeds")

## 8. CFD check of the hover point — OpenFOAM, rotating reference frame

Blade element theory says 1.17 N at the hover rpm. `aeromant`'s `rotor_mrf_static` template puts the
same CAD propeller in a rotating cell zone (MRF, steady k-ω SST) and integrates the blade forces.
It is a *check*, coarse by design (about 100 k cells, a few minutes on one core; `surface_level=4`
for a finer blade at several times the cost); expect tens of percent against BEMT, and read the sign
message: a negative thrust means the propeller is handed against `rotation`.

The propeller CAD has its axis along Z; the template wants it along +x, so the shape is rotated first.
Nothing runs unless you run the cell; `VEGETA_SKIP_OPENFOAM=1` skips it (used when the notebooks are
executed headlessly on a machine without OpenFOAM).

In [ ]:
prop_x = dedalus.Geometry.from_cadquery(cad.shape.rotate((0, 0, 0), (0, 1, 0), 90), name="prop_axis_x")   # axis z -> +x
stl_x = prop_x.export_stl(RUNS / "quad_5x43_cad" / "prop_axis_x.stl", tolerance=0.02)
CFD_PARAMS = dict(rpm=hover.rpm, diameter=d, kinematic_viscosity=1.5e-5, density=RHO, rotation=1, iterations=400,
                  cells_per_diameter=6.0, surface_level=3, near_level=2, rotor_level=2, wake_level=1)   # coarse: minutes, not hours
cfd = None
if RUN_CFD:
    case = aeromant.CFDCase("rotor_mrf_static", stl_x, CFD_PARAMS, workdir=RUNS / "quad_5x43_cfd_hover", geometry_units="mm",
                            environment=aeromant.OpenFOAMEnvironment.detect())
    print(case.prepare(overwrite=True))
    cfd = case.run(progress=True)
    print(cfd)
else:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1): run this cell on a machine with OpenFOAM to get the check")

In [ ]:
if cfd is not None and cfd.ok:
    m = cfd.metrics
    compare = pd.DataFrame({"BEMT (Boreas)": {"thrust_N": hover.thrust, "torque_Nm": hover.aero.torque, "power_W": hover.aero.power, "figure_of_merit": hover.aero.figure_of_merit},
                            "CFD (rotor_mrf_static)": {"thrust_N": m["thrust_N"], "torque_Nm": m["torque_Nm"], "power_W": m["power_W"], "figure_of_merit": m["figure_of_merit"]}})
    compare["CFD / BEMT"] = compare["CFD (rotor_mrf_static)"] / compare["BEMT (Boreas)"]
    print(f"{m['mesh_cells']} cells, {'converged' if m['converged'] else 'not converged'} in {m['iterations']} iterations; forces averaged over the last {m['averaging_window']}")
    display(compare.round(3))

In [ ]:
if cfd is not None and cfd.ok:
    aviz.show(aviz.plot_field_slice(case, "U", normal="z"))          # the plane through the axis: inflow above, slipstream below
    fig = aviz.plot_section(case, "p", normal="z", zoom=2)

In [ ]:
if cfd is not None and cfd.ok:
    aviz.show(aviz.plot_streamlines(case, n=80, normal_plane="z"))
    aviz.show(aviz.plot_surface_pressure(case))

### The flow as a video

Tracer particles carried through the converged velocity field (a steady result played as motion), the rotor turned at its rpm for the eye; written with OpenCV. Open the file with any player if the inline video does not show.

In [ ]:
if cfd is not None and cfd.ok:
    from IPython.display import Video
    video = aviz.animate_particles(case, RUNS / "quad_5x43_hover_flow.mp4", rpm=hover.rpm, seconds=6, fps=24)
    display(Video(str(video), embed=False, width=720))

## 9. The blade under load (Talos), and the animation

One blade with its hub, hub faces fixed, the blade's share of the full-throttle thrust and of the torque
(at 0.7 R) applied as tractions over the blade — the right totals with an approximate distribution.
A moulded glass-filled nylon blade (E = 8 GPa, yield 100 MPa) — an assumption to replace with the real material. The video ramps the load from zero to full while the blade turns at the full-throttle
rpm: a linear static result presented in motion, not a transient analysis.

In [ ]:
blade = PropellerCAD().generate(**dict(CAD_KW, blades=1))
bfiles = blade.export(RUNS / "quad_5x43_blade", formats=("step",))
hub_r, hub_h, R_tip = CAD_KW["hub_diameter"] / 2, CAD_KW["hub_height"], CAD_KW["diameter"] / 2
BLADE_REGIONS = [talos.SurfacesInBox("hub", (-hub_r - 0.5, -hub_r - 0.5, -hub_h / 2 - 0.5, hub_r + 0.5, hub_r + 0.5, hub_h / 2 + 0.5)),
                 talos.SurfacesInBox("blade", (hub_r - 1.5, -R_tip, -R_tip, R_tip + 1.0, R_tip, R_tip))]
BLADE_MAT = talos.Material("PA6-GF30 (moulded blade)", youngs_modulus=8000.0, poissons_ratio=0.35, density=1.35e-9, yield_strength=100.0, source="assumed moulded glass-filled nylon")
T_blade = punch.thrust / prop.blades                                     # N per blade at full throttle
F_tan = punch.aero.torque / (prop.blades * 0.7 * prop.radius)           # tangential force per blade at 0.7 R
blade_model = talos.StructuralModel(bfiles.artifacts["step"], "mm-N-MPa", BLADE_MAT, BLADE_REGIONS, [talos.FixedSupport("hub")],
                                    [talos.Force("blade", fz=T_blade, fy=-F_tan)], talos.MeshSettings(element_size=0.8), name="blade_full_throttle")
blade_mesh = blade_model.mesh(RUNS / "quad_5x43_blade_fea", progress=True)
res_blade = blade_model.solve(RUNS / "quad_5x43_blade_fea", progress=True)
print(f"per blade at full throttle ({punch.rpm:.0f} rpm): thrust {T_blade:.2f} N, tangential {F_tan:.2f} N")
print(res_blade)
if res_blade.ok:
    tviz.show(tviz.plot_results(res_blade, field="von_mises"))
    from IPython.display import Video
    video = tviz.animate(res_blade, RUNS / "quad_5x43_blade_stress.mp4", rpm=punch.rpm, axis="z", seconds=5, fps=24)
    display(Video(str(video), embed=False, width=720))

In [ ]:
if cfd is not None and cfd.ok:                                    # add the CFD point to the exported JSON
    doc = json.loads((RUNS / "quad_5x43.json").read_text())
    doc["cfd_hover"] = {"template": "rotor_mrf_static", "parameters": CFD_PARAMS, "metrics": {k: v for k, v in cfd.metrics.items() if not isinstance(v, (list, dict))}}
    (RUNS / "quad_5x43.json").write_text(json.dumps(doc, indent=2, default=float))
    print("cfd_hover added to", RUNS / "quad_5x43.json")

**Hand-off:** Part 3 takes `hover`/`cruise`/`punch` (rpm, thrust, unbalance) straight from these objects; the JSON is the same record for anything outside this notebook. Fit `airfoil` to a measured polar before trusting the absolute numbers.

# Part 3 — vibration, cyclic loads and life

In [ ]:
RUNS = ROOT / "life"; RUNS.mkdir()
PREFERRED = {k: preferred.params[k] for k in ("arm_height", "plate_thickness", "arm_width", "taper")}   # the preferred revision of Part 1
PROP = {name: {"rpm": pt.rpm, "thrust_N": pt.thrust, "unbalance_N": boreas.excitations(prop, pt.rpm)["unbalance_force_n"]}
        for name, pt in (("hover", hover), ("cruise", cruise), ("full", punch))}                    # Part 2 operating points
BLADES = prop.blades
MOTOR_PROP_MASS_T = (motor.mass_kg + prop.mass_kg) * 1e-3     # tonnes on each pad
STACK_MASS_T = (parts["mass_g"].sum() - MOTORS * (motor.mass_kg + prop.mass_kg) * 1000 - parts.loc["frame (printed, from CAD volume)", "mass_g"]) * 1e-6   # everything else, on the stack bolts
MAX_THRUST_N = MAX_THRUST_PER_MOTOR_N                          # the structural sizing value of Part 1
print(f"preferred frame {PREFERRED} | motor+prop {MOTOR_PROP_MASS_T * 1e6:.0f} g per pad | stack {STACK_MASS_T * 1e6:.0f} g")
pd.DataFrame(PROP).round(2)

## 1. The frame, its masses, and a mesh used by every analysis

The mesh is generated once and copied per load case: the unit cases and the modal analysis must live
on the *same* nodes for the superposition later.

In [ ]:
p = frame_design.resolve(**PREFERRED)
frame = frame_design.generate(**PREFERRED)
cad = frame.export(RUNS / "cad")            # PETG_CF: the material of Part 1

def frame_regions(p):
    R, m, r = p["wheelbase"] / 2, p["motor_pattern"] / 2, p["motor_hole"] / 2 + 0.2
    regions = []
    for k in range(4):
        a = math.radians(45 + 90 * k)
        cx, cy = R * math.cos(a), R * math.sin(a)
        regions.append(talos.SurfacesInBox(f"motor{k}", (cx - m - r, cy - m - r, -0.1, cx + m + r, cy + m + r, 100.0)))
    s, r = p["stack_pattern"] / 2, p["stack_hole"] / 2 + 0.2
    for k, (sx, sy) in enumerate([(s, s), (-s, s), (s, -s), (-s, -s)]):
        regions.append(talos.SurfacesInBox(f"stack{k}", (sx - r, sy - r, -0.1, sx + r, sy + r, 100.0)))
    return regions

REGIONS = frame_regions(p)
SUPPORTS = [talos.FixedSupport(f"stack{k}") for k in range(4)]
MASSES = [talos.PointMass(f"motor{k}", MOTOR_PROP_MASS_T) for k in range(4)] + \
         [talos.PointMass(f"stack{k}", STACK_MASS_T / 4) for k in range(4)]
MESH = talos.MeshSettings(element_size=5.0)

def model(loads, name):
    return talos.StructuralModel(cad.artifacts["step"], "mm-N-MPa", PETG_CF, REGIONS, SUPPORTS, loads, MESH, name=name, masses=MASSES)

base = model([], "modal")
mesh_res = base.mesh(RUNS / "mesh", progress=True)
print(mesh_res)

def case_dir(name):                         # a copy of the meshed directory per analysis
    d = RUNS / name
    if not d.exists():
        shutil.copytree(RUNS / "mesh", d)
    return d

## 2. Natural frequencies with the motors and the stack on board

The pads carry 35 g each (motor + propeller), the stack bolts 240 g. Without those masses the arm
frequencies would come out several times too high.

In [ ]:
modes = base.solve_modes(case_dir("modal"), n_modes=8, progress=True)
print(modes)
freqs = modes.metrics["frequencies_hz"]

In [ ]:
tviz.show(tviz.plot_mode(modes, mode=1))     # arm bending (vertical), motor mass at the tip

In [ ]:
tviz.show(tviz.plot_mode(modes, mode=5))     # in-plane arm bending: what the unbalance force excites

### Campbell diagram: where the rotor lines cross the frame modes

The frequency diagram: shaft (1P) and blade-pass (3P) lines against rpm, the frame modes as horizontal lines, the operating points marked. A crossing near an operating rpm is a resonance to design out or to damp.

In [ ]:
structure = chronos.Structure(tuple(freqs), damping_ratio=0.03, source="Talos modal, PETG-CF nominal E, lumped masses")
rpm_ops = {k: v["rpm"] for k, v in PROP.items()}
fig = structure.campbell({"1P": 1, "3P": BLADES}, np.linspace(1000, 30000, 30), operating_rpm=rpm_ops)
lines = {k: v["rpm"] / 60 for k, v in PROP.items()}
pd.DataFrame({k: {"1P_hz": f, "nearest_mode_hz": structure.nearest_mode(f), "margin": structure.margin(f),
                  "amplification_1P": float(structure.amplification(f)[0])} for k, f in lines.items()}).round(2)

## 3. Unit load cases — the stress "per newton" of each load pattern

Three patterns, each a normal linear static run at a known load:

| pattern | unit case | level unit |
|---|---|---|
| `thrust` | +8 N up on every motor's bolt holes | N per motor |
| `unbalance` | 1 N horizontal at motor 0 (a rotating unbalance seen by the arm) | N |
| `landing` | +40 N up on motor 0 only | N |

Because the analysis is linear, the stress at any level is level × unit stress — that is what lets
a mission with 10⁵ vibration cycles be evaluated without 10⁵ solves.

In [ ]:
UNIT = {
    "thrust":    (model([talos.Force(f"motor{k}", fz=MAX_THRUST_N) for k in range(4)], "thrust"), MAX_THRUST_N),
    "unbalance": (model([talos.Force("motor0", fx=1.0)], "unbalance"), 1.0),
    "landing":   (model([talos.Force("motor0", fz=40.0)], "landing"), 40.0),
}
unit_results = {}
for name, (m, load) in tqdm(UNIT.items(), desc="unit cases"):
    unit_results[name] = m.solve(case_dir(name), progress=False)
    r = unit_results[name]
    print(f"{name:<10} {'ok' if r.ok else 'FAILED'}  max von Mises {r.metrics.get('max_von_mises', float('nan')):.2f} MPa at {load:g} N")
unit_cases = {k: (unit_results[k], UNIT[k][1]) for k in UNIT}

In [ ]:
tviz.show(tviz.plot_results(unit_results["unbalance"], field="von_mises"))   # 1 N sideways on one pad

## 4. Three missions

Levels are **per-motor thrust in N** (`thrust`), the rotating force in N (`unbalance`) and the one-arm
landing force in N (`landing`). The unbalance is ISO G 6.3 from notebook 11. `repeat` makes a segment
an excursion from the previous level, so every punch-out closes a load cycle.

In [ ]:
def unb(point, rpm=None, force=None):
    rpm = rpm or PROP[point]["rpm"]; force = force if force is not None else PROP[point]["unbalance_N"]
    return chronos.Excitation(f"unbalance {point}", rpm / 60, force, "unbalance")

SPOOL = chronos.Excitation("unbalance spool-up", 4500 / 60, 0.06, "unbalance")     # passes through the arm modes
HOVER, CRUISE = PROP["hover"]["thrust_N"], PROP["cruise"]["thrust_N"]

missions = {
    "inspection": chronos.Mission("inspection", (
        chronos.Segment("spool-up", 3, {"thrust": 0.5}, (SPOOL,)),
        chronos.Segment("take-off", 8, {"thrust": 2.0}, (unb("hover"),)),
        chronos.Segment("hover + slow moves", 660, {"thrust": HOVER}, (unb("hover"),)),
        chronos.Segment("position changes", 4, {"thrust": 1.8}, (unb("cruise"),), repeat=12),
        chronos.Segment("landing", 3, {"thrust": 0.6, "landing": 12.0}),
    ), "12 min structure inspection: mostly hover, gentle moves, soft landing"),
    "freestyle": chronos.Mission("freestyle", (
        chronos.Segment("spool-up", 3, {"thrust": 0.5}, (SPOOL,)),
        chronos.Segment("hover", 60, {"thrust": HOVER}, (unb("hover"),)),
        chronos.Segment("punch-out", 1.5, {"thrust": MAX_THRUST_N}, (unb("full", force=0.38),), repeat=40),
        chronos.Segment("hard turns", 2.0, {"thrust": 4.0}, (unb("cruise", rpm=15000, force=0.20),), repeat=60),
        chronos.Segment("cruise between tricks", 120, {"thrust": CRUISE}, (unb("cruise"),)),
        chronos.Segment("hard landing", 2, {"thrust": 0.8, "landing": 40.0}),
    ), "5 min freestyle: 40 punch-outs, 60 hard turns, one hard landing"),
    "cruise": chronos.Mission("cruise", (
        chronos.Segment("spool-up", 3, {"thrust": 0.5}, (SPOOL,)),
        chronos.Segment("climb", 20, {"thrust": 2.5}, (unb("cruise"),)),
        chronos.Segment("cruise out", 420, {"thrust": CRUISE}, (unb("cruise"),)),
        chronos.Segment("gust corrections", 2.0, {"thrust": 2.4}, (unb("cruise"),), repeat=30),
        chronos.Segment("cruise back", 420, {"thrust": CRUISE}, (unb("cruise"),)),
        chronos.Segment("landing", 3, {"thrust": 0.6, "landing": 25.0}),
    ), "15 min tilted cruise out and back with gust corrections"),
}
for m in missions.values():
    fig = m.profile(patterns=["thrust", "landing"])
pd.DataFrame({k: {"duration_min": m.duration_h * 60, "segments": len(m.segments), "patterns": ", ".join(m.patterns)} for k, m in missions.items()}).T

## 5. Load spectra: rainflow for the manoeuvres, amplified vibration for the rotor

`build_spectrum` counts the manoeuvre cycles of each pattern (ASTM rainflow on the level sequence) and
adds one block per segment and excitation: cycles = frequency × time, amplitude = force × dynamic
amplification from the nearest frame mode. Look at the spool-up block: small force, large factor.

In [ ]:
spectra = {k: chronos.build_spectrum(m, structure) for k, m in missions.items()}
for k, sp in spectra.items():
    sp.save(RUNS / f"spectrum_{k}.json")
    fig = sp.plot()
spectra["freestyle"].table().round(4)

## 6. Damage per mission on the whole frame (Talos), and the hotspot

S-N curve for PETG-CF — **assumed**: σ_f = 80 MPa, b = −0.11, Goodman with 55 MPa ultimate. Printed
polymers scatter a lot; a few coupons in the print orientation of the arms would replace these numbers.

In [ ]:
CURVE = talos.FatigueCurve("PETG-CF (assumed)", sigma_f=80.0, b=-0.11, ultimate=55.0,
                           source="assumed Basquin fit; replace with coupon tests")
fatigue = {}
for k in tqdm(missions, desc="fatigue"):
    fatigue[k] = talos.assess_fatigue(unit_cases, spectra[k].to_dict(), CURVE, workdir=RUNS / f"fatigue_{k}")
life = pd.DataFrame({k: {"duration_min": missions[k].duration_h * 60, "damage_per_mission": f.result.metrics["damage_per_pass"],
                         "missions_to_failure": f.result.metrics["passes_to_failure"],
                         "hours_to_failure": f.result.metrics["hours_to_failure"],
                         "hotspot": tuple(round(x, 1) for x in f.result.metrics["hotspot_location"])} for k, f in fatigue.items()}).T
life

In [ ]:
worst = life["damage_per_mission"].astype(float).idxmax()
tviz.show(tviz.plot_damage(fatigue[worst], unit_results["thrust"].artifacts["mesh"]))

In [ ]:
contrib = pd.Series(fatigue[worst].contributions).sort_values(ascending=False)
ax = contrib.head(8).plot.barh(figsize=(8, 3.2), title=f"{worst}: damage contributions at the hotspot"); ax.invert_yaxis()
contrib.head(8).round(6)

## 7. Back to the structural calculation: the worst combined static state

The spectrum also tells which *combination* of levels is the worst static case in each mission
(maximum thrust with the landing load never coincide; the peak of each pattern does with the vibration
amplitude). By superposition the peak stress at the hotspot is Σ level × unit stress: a static
safety-factor check that includes the vibration — this is the "pass it back to the FEM" step.

In [ ]:
hot = fatigue[worst].hotspot
unit_vm = {k: float(talos.read_frd(r.artifacts["frd"]).von_mises[hot]) / UNIT[k][1] for k, r in unit_results.items()}
rows = {}
for k, sp in spectra.items():
    peak = {}
    for b in sp.blocks:
        peak[b.pattern] = max(peak.get(b.pattern, 0.0), abs(b.mean) + abs(b.amplitude))
    stress = sum(peak.get(pat, 0.0) * unit_vm[pat] for pat in unit_vm)
    rows[k] = {**{f"peak_{pat}": peak.get(pat, 0.0) for pat in unit_vm}, "hotspot_stress_MPa": stress,
               "SF_yield": PETG_CF.yield_strength / stress}
pd.DataFrame(rows).T.round(2)

## 8. Long-term: a fleet usage over thousands of flights

Damage per mission is combined over a usage mix (drawn at random, seeded) until Miner's sum reaches 1.
Two other mixes and a balanced-propeller variant (G 2.5 instead of G 6.3: unbalance force ÷ 2.5) show
what actually drives the life.

In [ ]:
damage = {k: f.result.metrics["damage_per_pass"] for k, f in fatigue.items()}
hours = {k: m.duration_h for k, m in missions.items()}
usage = {"inspection": 0.6, "freestyle": 0.15, "cruise": 0.25}
sim = chronos.simulate_life(damage, hours, usage, n_flights=40000, seed=0)
fig = sim.plot()
print(f"failure after {sim.flights_to_failure:.0f} flights / {sim.hours_to_failure:.0f} h with usage {usage}")

In [ ]:
def life_hours(dmg, mix):
    return chronos.simulate_life(dmg, hours, mix, n_flights=200000, seed=None).hours_to_failure

balanced = {}
for k, sp in spectra.items():
    sp2 = chronos.LoadSpectrum(sp.mission, sp.duration_s,
                               [chronos.Block(b.pattern, b.mean, b.amplitude / 2.5 if b.pattern == "unbalance" else b.amplitude, b.cycles, b.source) for b in sp.blocks],
                               sp.patterns)
    balanced[k] = talos.assess_fatigue(unit_cases, sp2.to_dict(), CURVE).result.metrics["damage_per_pass"]
mixes = {"inspection-heavy": {"inspection": 0.6, "freestyle": 0.15, "cruise": 0.25},
         "freestyle-heavy":  {"inspection": 0.2, "freestyle": 0.6,  "cruise": 0.2},
         "cruise-only":      {"inspection": 0.0, "freestyle": 0.0,  "cruise": 1.0}}
pd.DataFrame({name: {"hours, G 6.3 props": life_hours(damage, mix), "hours, balanced G 2.5": life_hours(balanced, mix)}
              for name, mix in mixes.items()}).T.round(0)

## 9. Export and record

In [ ]:
summary = {
    "design": {"file": "designs/quad_frame.py", "parameters": p},
    "modes_hz": freqs, "damping_ratio": structure.damping_ratio,
    "excitations_hz": lines, "unit_cases": {k: {"load": UNIT[k][1], "max_von_mises": unit_results[k].metrics["max_von_mises"]} for k in UNIT},
    "curve": CURVE.__dict__, "missions": {k: m.describe() for k, m in missions.items()},
    "damage_per_mission": damage, "hours_per_mission": hours,
    "usage": usage, "hours_to_failure": sim.hours_to_failure, "flights_to_failure": sim.flights_to_failure,
    "spectra": {k: str(RUNS / f"spectrum_{k}.json") for k in missions},
}
(RUNS / "life.json").write_text(json.dumps(summary, indent=2, default=float))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**What to do with this:** the hotspot and the dominant blocks say *where* and *why* the frame will
crack; the balanced-propeller row says whether balancing is worth more than material; the static
re-check says whether the worst mission ever gets near yield. Next revisions (notebook 08's workspace,
or a campaign in notebook 10 with `hours_to_failure` as the objective) close the loop.